In [ ]:
!pip install git+https://github.com/huggingface/transformers accelerate
!pip install qwen-vl-utils[decord]==0.0.8
!pip install flash_attn-2.7.4.post1+cu12torch2.6cxx11abiTRUE-cp39-cp39-linux_x86_64.whl

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-0ed7r2tv
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-0ed7r2tv
  Resolved https://github.com/huggingface/transformers to commit 9ed801f3ef0029e3733bbd2c9f9f9866912412a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.5/534.5 kB 16.9 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.0.0.dev0-py3-none-any.whl size=11155169 sha256=b7bc3022c882f181fa67b8d35f0e99f11c2484bf3c5027433f409550ec2808e8
  Stored in directory: /tmp/pip-ephem-wheel-cache-f0xd3xvt/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoProcessor, Glm4vForConditionalGeneration
import torch

MODEL_PATH = "zai-org/GLM-4.6V-Flash"
processor = AutoProcessor.from_pretrained(MODEL_PATH, use_fast=True)
model = Glm4vForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/369 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/704 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
import torch
from transformers import AutoProcessor, Glm4vForConditionalGeneration



def inference_glm_with_budget(model, processor, image_path, prompt, thinking_budget, max_new_tokens):
    """
    Runs inference with the GLM-4.6V model using a "thinking budget".

    This function performs a two-step generation to avoid reasoning loops.
    Returns only the model's final answer, excluding the thinking content.

    Args:
        model: The loaded Glm4vForConditionalGeneration model.
        processor: The AutoProcessor matching the model.
        image_path (str): Web URL or local path of the image.
        prompt (str): The text prompt for the model.
        thinking_budget (int): Maximum number of tokens for the reasoning phase.
        max_new_tokens (int): Maximum total number of tokens to generate.

    Returns:
        str: The model's final answer (text after </think>).
    """

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "url": image_path
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(model.device)



    input_length = inputs["input_ids"].shape[1]

    generated_ids_step1 = model.generate(
        **inputs, max_new_tokens=thinking_budget,
        do_sample = False,
        #repetition_penalty=1.3
    )
    output_ids_step1 = generated_ids_step1[0][input_length:].tolist()

    end_token_id = processor.tokenizer.eos_token_id
    end_think_token_id = processor.tokenizer.convert_tokens_to_ids('</think>')
    input_ids_for_second_gen_tensor = generated_ids_step1
    needs_second_generation = True

    if end_token_id in output_ids_step1:
        needs_second_generation = False
    elif end_think_token_id not in output_ids_step1:
        early_stopping_text = (
            "\n</think>\n"
            "ANALYSIS COMPLETE.\n"
            "Now, provide the final, detailed textual description of the circuit based on the analysis.\n"
            "Do not add further commentary or self-correction. "
            "The output must be the description ONLY.\n"
        )
        early_stopping_ids = processor.tokenizer(
            early_stopping_text, return_tensors="pt", add_special_tokens=False
        ).input_ids.to(model.device)
        input_ids_for_second_gen_tensor = torch.cat([generated_ids_step1, early_stopping_ids], dim=-1)

    final_generated_ids = input_ids_for_second_gen_tensor
    if needs_second_generation:
        num_tokens_so_far = input_ids_for_second_gen_tensor.shape[1] - input_length
        remaining_tokens = max_new_tokens - num_tokens_so_far
        if remaining_tokens > 0:
            inputs_for_second_gen = inputs.copy()
            inputs_for_second_gen['input_ids'] = input_ids_for_second_gen_tensor
            inputs_for_second_gen['attention_mask'] = torch.ones_like(input_ids_for_second_gen_tensor)

            final_generated_ids = model.generate(
                **inputs_for_second_gen,
                max_new_tokens=remaining_tokens,
                do_sample = False,
                #repetition_penalty=1.3
            )

    # Debug print
    full_generated_text = processor.decode(final_generated_ids[0][input_length:], skip_special_tokens=True).strip()
    print("-" * 50)
    print(">>> DEBUG: FULL OUTPUT GENERATED BY THE MODEL <<<")
    print(full_generated_text)
    print("-" * 50)

    final_output_ids = final_generated_ids[0][input_length:].tolist()
    try:
        index_end_think = len(final_output_ids) - final_output_ids[::-1].index(end_think_token_id)
    except ValueError:
        index_end_think = 0

    final_answer = processor.decode(
        final_output_ids[index_end_think:],
        skip_special_tokens=True
    ).strip()

    return final_answer

In [ ]:
import os, json
from PIL import Image


image_directory = "/content/drive/MyDrive/netlist_test"
output_directory = "results/" # Directory where the text files are saved

os.makedirs(output_directory, exist_ok=True)


prompt = """
You are an expert circuit analyst. Your task is to analyze the provided circuit schematic and provide a detailed, step-by-step textual description of its topology. Your analysis must be meticulous and follow a strict protocol to ensure accuracy.

**OUTPUT FORMAT:**
- The output MUST BE ONLY a clear, well-structured textual description in plain language.
- Do NOT output JSON, code, or use markdown blocks.

---
**CORE ANALYSIS PRINCIPLES**
These are the fundamental rules you MUST apply throughout your entire analysis.

1.  **MOSFET Pin Identification:** A MOSFET symbol has one pin on one side and two on the other.
    - The **Gate** is the single pin on its own side.
    - The **Source** is the pin with an arrow on the other side.
    - The **Drain** is the remaining pin without an arrow.

2.  **Connection Rule (Dot Convention):** Two wires crossing are connected ONLY if there is a solid dot at the intersection. If there is no dot, the wires cross without connecting.

---
**STRICT ANALYSIS PROTOCOL**
You MUST follow these steps in this exact order.

**PHASE 1: Global Identification and Verification of Nets**
1.  **List Labeled Nets:** First, create a list of all clearly labeled nets (e.g., Vin, Vout, VDD). Use '0' for the ground net (GND, VSS, or ground symbol).
2.  **Identify and Verify Unlabeled Nodes:** Next, locate any important unlabeled nodes (junctions of multiple components). For EACH such node, you MUST perform this check:
    - Is this node directly connected by a wire (with no components in between) to any of the labeled nets from step 1?
    - If YES, that node IS PART OF the existing net. Do NOT create a new name for it.
    - If NO, and only if it's a truly separate electrical point, it is a new, distinct net.
3.  **Assign Names and Finalize Inventory:** Assign unique, sequential names (`n1`, `n2`, etc.) ONLY to the new, distinct nets identified in step 2. The final output for this phase MUST be a complete inventory of ALL verified nets.

**PHASE 2: Description of Components and Connections**
- After the net inventory, list each component you identify one by one.
- For each component:
    - Specify its ID and type.
    - **For MOSFETs ONLY:** Include an "Analysis:" section where you re-state and apply the core pin identification logic.
    - **For ALL components:** Conclude with a section explicitly labeled "**Final Connections:**" containing a bullet-point list.
    - **Consistency Rule:** You MUST use the net names from the final inventory in PHASE 1. Do NOT use relative descriptions like "the node between X and Y".

---
**EXAMPLE OF CORRECT EXECUTION:**
*   **For a schematic showing:** A cascode M1/M2. M1 drain is connected to M2 source. This node is unlabeled but also connects to a resistor going to VDD.
*   **A perfect output following the protocol would be:**

**PHASE 1: Global Identification and Verification of Nets**
The primary labeled nets are VDD, Vout, Vin, Vb, and 0 (ground).
An unlabeled node connecting the drain of M1 and the source of M2 was identified. A check confirms it is NOT directly connected to any other labeled net. Therefore, it is assigned the new name n1.
The final net inventory is: VDD, Vout, Vin, Vb, 0, n1.

**PHASE 2: Description of Components and Connections**
- **Component M1:** This is an NMOS transistor.
    - **Analysis:**
        - Pin identification logic: A MOSFET symbol has one pin on one side and two on the other. The single pin on its own side is the Gate. Of the two pins on the other side, the one with an arrow is the Source. The remaining pin is the Drain.
        - Pin Mapping: For M1, applying this logic, I have identified: Gate is connected to Vin. Source is connected to 0. Drain is connected to n1.
    - **Final Connections:**
        - Its **gate** is connected to the 'Vin' node.
        - Its **source** and **bulk** are connected to 0 (ground).
        - Its **drain** is connected to the 'n1' node.
... (and so on for the other components)

---
Now, analyze the following image and generate the detailed textual description by STRICTLY following all principles and the two-phase protocol above.
"""





for filename in os.listdir(image_directory):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
        base_filename = os.path.splitext(filename)[0]
        image_path = os.path.join(image_directory, filename)
        print(image_path)

        response = inference_glm_with_budget(model, processor, image_path, prompt, 6000, 8192)
        print(response)




        output_filename = f"{base_filename}.txt"
        output_path = os.path.join(output_directory, output_filename)

        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(response)

        print(f"Response saved to: {output_path}\n")



/content/drive/MyDrive/netlist_test/Multivibrator.jpg


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


--------------------------------------------------
>>> DEBUG: FULL OUTPUT GENERATED BY THE MODEL <<<
<think>Got it, let's tackle this step by step. First, I need to go through Phase 1: Global Identification and Verification of Nets.

First, list labeled nets. Looking at the schematic: there's a 9V+ battery, so that's a positive supply. Then there's a ground symbol at the bottom left and bottom right, which is 0 (ground). The components have labels like R1, R2, R3, R4, L1, L2, C1, C2, Q1, Q2. So labeled nets are: 9V+ (but wait, the battery is a voltage source, so the positive terminal is connected to the top left node, and the negative terminal is ground (0). So labeled nets are: 9V+ (the positive terminal of the battery) and 0 (ground). Are there any other labeled nets? The diagram has nodes labeled with component connections, but the main labeled ones are the battery's positive and ground. So step 1: List Labeled Nets: 9V+, 0.

Next, step 2: Identify and verify unlabeled nodes. Let's 